In [44]:
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import ( OrdinalEncoder, OneHotEncoder,LabelEncoder )
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import ( GridSearchCV,
RandomizedSearchCV,
StratifiedKFold,
cross_val_score)

from sklearn.metrics import (accuracy_score,precision_score, f1_score, recall_score , confusion_matrix, classification_report, ConfusionMatrixDisplay)


In [46]:
data = pd.read_csv('loan.csv')
data

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...,...
609,LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
610,LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
611,LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y
612,LP002984,Male,Yes,2,Graduate,No,7583,0.0,187.0,360.0,1.0,Urban,Y


In [48]:
data = data.drop('Loan_ID', axis=1)
data.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [50]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Gender             601 non-null    object 
 1   Married            611 non-null    object 
 2   Dependents         599 non-null    object 
 3   Education          614 non-null    object 
 4   Self_Employed      582 non-null    object 
 5   ApplicantIncome    614 non-null    int64  
 6   CoapplicantIncome  614 non-null    float64
 7   LoanAmount         592 non-null    float64
 8   Loan_Amount_Term   600 non-null    float64
 9   Credit_History     564 non-null    float64
 10  Property_Area      614 non-null    object 
 11  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(7)
memory usage: 57.7+ KB


In [52]:
le=LabelEncoder()
data['Loan_Status'] =le.fit_transform(data['Loan_Status'])
data.head()

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,1
1,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,0
2,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,1
3,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,1
4,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,1


In [54]:
x = data.drop('Loan_Status',axis= 1)
y= data['Loan_Status']


In [56]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state= 42 , test_size= 0.3)

In [58]:
num_cols = x.select_dtypes(include = ['int64','float64']).columns
cat_cols = x.select_dtypes(include = ['object']).columns

In [60]:
num_pipeline = Pipeline ([
    ('imputer',SimpleImputer (strategy = 'mean')),
    ('Scaler',StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy = 'most_frequent')),
    ('Encoding',OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer([
    ('num',num_pipeline,num_cols),
    ('cat',cat_pipeline,cat_cols)
])
 
pipe = Pipeline ([
    ('preprocessing', preprocessor ),
    ('model',KNeighborsClassifier())
])
pipe

pipe.fit(x_train,y_train)
pred = pipe.predict(x_test)
print("accuracy:", accuracy_score(y_test, pred))
print("pression:", precision_score(y_test, pred))
print("f1 score:", f1_score(y_test, pred))
print("recall score:", recall_score(y_test, pred))
print(classification_report (y_test, pred))
print(confusion_matrix(y_test,pred))

accuracy: 0.772972972972973
pression: 0.7532467532467533
f1 score: 0.8467153284671532
recall score: 0.9666666666666667
              precision    recall  f1-score   support

           0       0.87      0.42      0.56        65
           1       0.75      0.97      0.85       120

    accuracy                           0.77       185
   macro avg       0.81      0.69      0.70       185
weighted avg       0.79      0.77      0.75       185

[[ 27  38]
 [  4 116]]


Cross validation CV

In [65]:

cv_score = cross_val_score (
    pipe, x, y, cv=5)
print(print("Cross Validation Scores :", cv_score))
print(cv_score.mean())

Cross Validation Scores : [0.79674797 0.7398374  0.76422764 0.79674797 0.80327869]
None
0.7801679328268692


In [67]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

GRID SEARCH CV

In [70]:
param_grid = {

    'model__n_neighbors': [1,3,5,7,9],

    'model__weights': ['uniform', 'distance'],

    'model__leaf_size': [20,30,40],

    'model__p': [1,2],

    'model__metric': ['minkowski', 'euclidean', 'manhattan']

}
param_grid


import time
grid = GridSearchCV(
    estimator = pipe,
    param_grid = param_grid,
    cv=cv,
    n_jobs = 1
)
start_time = time.time()
grid.fit(x_train,y_train)
end_time = time.time()

print("\nGRID SEARCH COMPLETED")
print("Execution Time :", round(end_time - start_time, 3),"seconds")


GRID SEARCH COMPLETED
Execution Time : 18.532 seconds


In [71]:
import time
grid = GridSearchCV(
    estimator = pipe,
    param_grid = param_grid,
    cv=cv,
    n_jobs = 1
)
start_time = time.time()
grid.fit(x_train,y_train)
end_time = time.time()

print("\nGRID SEARCH COMPLETED")
print("Execution Time :", round(end_time - start_time, 3),"seconds")

#  RANDOMIZED SEARCH CV
print("\nBEST PARAMETERS")
print(grid.best_params_)

print("\nBEST SCORE")
print(grid.best_score_)


GRID SEARCH COMPLETED
Execution Time : 18.641 seconds

BEST PARAMETERS
{'model__leaf_size': 20, 'model__metric': 'minkowski', 'model__n_neighbors': 9, 'model__p': 2, 'model__weights': 'uniform'}

BEST SCORE
0.8065663474692203


RANDOMIZED SEARCH CV 

In [73]:
random_search = RandomizedSearchCV (
    estimator = pipe,
    param_distributions = param_grid ,
    cv= cv,
    n_iter = 50 )

random_search.fit(x_train,y_train)
print("\n Randomized excuted successfuly ")


print("|n BEST PARAMETERS ")
print(random_search.best_params_)


print("\n BEST SCORE ")
print(random_search.best_score_)


 Randomized excuted successfuly 
|n BEST PARAMETERS 
{'model__weights': 'uniform', 'model__p': 2, 'model__n_neighbors': 9, 'model__metric': 'minkowski', 'model__leaf_size': 20}

 BEST SCORE 
0.8065663474692203
